In [0]:
import json
import pandas as pd
from pyspark.sql import functions as F

# 1. Leer Bronze
bronze = spark.table(
    "open_meteo.api_open.bronze_weather_raw"
)

# 2. Obtener el JSON de la última extracción
row = bronze.orderBy(
    F.col("extracted_at").desc()
).first()

data = json.loads(row["payload"])
hourly = data["hourly"]

# 3. Convertir los datos en una tabla
pdf = pd.DataFrame({
    "weather_time": hourly["time"],
    "temperature_c": hourly["temperature_2m"],
    "humidity_pct": hourly["relative_humidity_2m"],
    "precipitation_mm": hourly["precipitation"],
    "wind_speed_kmh": hourly["wind_speed_10m"]
})

df = spark.createDataFrame(pdf)

# 4. Limpiar los datos
df = (
    df
    .withColumn("weather_time", F.to_timestamp("weather_time"))
    .dropDuplicates(["weather_time"])
    .filter(F.col("weather_time").isNotNull())
)

# 5. Guardar Silver
df.write.format("delta").mode("overwrite").saveAsTable(
    "open_meteo.api_open.silver_weather_hourly"
)

display(df)